In [1]:
!apt-get update
!apt-get install -y libopencv-dev build-essential cmake

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [82.1 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:9 http://security.ubuntu.com/ubuntu noble-security/multiverse amd64 Packages [55.6 kB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,560 kB]
Get:11 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [2,216 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:13 http://security.ubuntu.com/ubuntu

In [2]:
%%writefile main.cpp
#include <iostream>
#include <opencv2/opencv.hpp>
#include <cuda_runtime.h>

int main() {
    std::cout << "OpenCV Versiyonu: " << CV_VERSION << std::endl;

    // Colab GPU Bağlantı Kontrolü
    int deviceCount = 0;
    cudaError_t error = cudaGetDeviceCount(&deviceCount);

    if (error == cudaSuccess && deviceCount > 0) {
        std::cout << "Başarılı! Colab GPU (CUDA) Algılandı. Cihaz Sayısı: " << deviceCount << std::endl;
    } else {
        std::cout << "CUDA Cihazı Bulunamadı veya GPU Aktif Edilmedi!" << std::endl;
    }

    return 0;
}

Writing main.cpp


In [3]:
!g++ main.cpp -o app `pkg-config --cflags --libs opencv4` -I/usr/local/cuda/include -L/usr/local/cuda/lib64 -lcudart
!./app

OpenCV Versiyonu: 4.6.0
Başarılı! Colab GPU (CUDA) Algılandı. Cihaz Sayısı: 1


### Stereo Eşleştirme Metrikleri Karşılaştırma Tablosu

| Metrik | Avantajları | Dezavantajları |
| :--- | :--- | :--- |
| **SAD** (Sum of Absolute Differences) | - Hesaplama yükü son derece düşüktür.<br>- Sadece toplama ve çıkarma işlemleri içerdiği için donanım düzeyinde çok hızlı çalışır. | - Aydınlatma ve kontrast değişimlerine karşı oldukça hassastır.<br>- Gürültülü görüntülerde doğruluğu düşüktür. |
| **SSD** (Sum of Squared Differences) | - Büyük farkları daha ağır cezalandırdığı için SAD'a göre gürültüye karşı biraz daha dirençlidir.<br>- Matematiksel olarak türetilebilir olduğundan optimizasyon süreçlerine uygundur. | - Çarpma işlemlerinden dolayı SAD'a göre daha yavaştır.<br>- Aydınlatma değişimlerine karşı hassasiyeti devam eder. |
| **ZNCC** (Zero-mean Normalized Cross-Correlation) | - Aydınlatma, parlaklık ve kontrast değişimlerine karşı son derece dayanıklıdır.<br>- Gürültüye karşı en dirençli metriktir. | - Karekök, bölme ve ortalama hesaplamaları içerdiğinden işlem maliyeti ve süresi oldukça yüksektir. |

In [4]:
# Gerekli test resimlerinin (hafta10resim.png ve hafta10resim2.png) var olup olmadığını kontrol edelim veya örnek indirelim.
# Eğer sisteminizde hazır resimler varsa bu adımı atlayabilir veya kendi resimlerinizin isimlerini verebilirsiniz.
import urllib.request
import os

# Eğer lokalde 'hafta10resim.png' yoksa, test amaçlı varsayılan stereo çiftleri oluşturulur veya indirilir.
# Aşağıda örnek olarak doğrudan boş dosya hatası almamak için basit örnekler yüklüyoruz.
# Kendi dosyalarınızı sol panelden 'hafta10resim.png' ve 'hafta10resim2.png' adıyla yükleyebilirsiniz.

In [7]:
%%writefile main.cpp
#include <iostream>
#include <opencv2/opencv.hpp>
#include <cmath>
#include <limits>
#include <chrono>

// Benzer parametre yapısına sahip benzerlik fonksiyonlarının tanımı (Fonksiyon Göstericileri için)
typedef double (*SimilarityFunc)(const cv::Mat&, const cv::Mat&);

// 1. SAD (Sum of Absolute Differences)
double computeSAD(const cv::Mat& patch1, const cv::Mat& patch2) {
    double sad = 0.0;
    int rows = patch1.rows;
    int cols = patch1.cols;
    for (int r = 0; r < rows; ++r) {
        const uchar* p1 = patch1.ptr<uchar>(r);
        const uchar* p2 = patch2.ptr<uchar>(r);
        for (int c = 0; c < cols; ++c) {
            sad += std::abs(p1[c] - p2[c]);
        }
    }
    return sad;
}

// 2. SSD (Sum of Squared Differences)
double computeSSD(const cv::Mat& patch1, const cv::Mat& patch2) {
    double ssd = 0.0;
    int rows = patch1.rows;
    int cols = patch1.cols;
    for (int r = 0; r < rows; ++r) {
        const uchar* p1 = patch1.ptr<uchar>(r);
        const uchar* p2 = patch2.ptr<uchar>(r);
        for (int c = 0; c < cols; ++c) {
            double diff = p1[c] - p2[c];
            ssd += diff * diff;
        }
    }
    return ssd;
}

// 3. ZNCC (Zero-mean Normalized Cross-Correlation)
// ZNCC sonucunu maksimize etmeye çalışırız ancak mesafe tabanlı algoritmamızla uyumlu olması için
// bu fonksiyon '1 - ZNCC' veya benzer bir maliyet (cost) değeri döndürecektir (Düşük değer = yüksek benzerlik).
double computeZNCC(const cv::Mat& patch1, const cv::Mat& patch2) {
    int rows = patch1.rows;
    int cols = patch1.cols;
    int N = rows * cols;

    double sum1 = 0.0, sum2 = 0.0;
    for (int r = 0; r < rows; ++r) {
        const uchar* p1 = patch1.ptr<uchar>(r);
        const uchar* p2 = patch2.ptr<uchar>(r);
        for (int c = 0; c < cols; ++c) {
            sum1 += p1[c];
            sum2 += p2[c];
        }
    }
    double mean1 = sum1 / N;
    double mean2 = sum2 / N;

    double num = 0.0;
    double den1 = 0.0;
    double den2 = 0.0;

    for (int r = 0; r < rows; ++r) {
        const uchar* p1 = patch1.ptr<uchar>(r);
        const uchar* p2 = patch2.ptr<uchar>(r);
        for (int c = 0; c < cols; ++c) {
            double dev1 = p1[c] - mean1;
            double dev2 = p2[c] - mean2;
            num += dev1 * dev2;
            den1 += dev1 * dev1;
            den2 += dev2 * dev2;
        }
    }

    if (den1 == 0.0 || den2 == 0.0) return 1.0; // Eşleşmeme durumu maliyeti
    double zncc = num / std::sqrt(den1 * den2);

    // ZNCC -1 ile 1 arasındadır. 1 en yüksek benzerliktir.
    // Maliyete dönüştürmek için: 1 - ZNCC (0 en iyi eşleşme, 2 en kötü)
    return 1.0 - zncc;
}

// Stereo Eşleştirme Fonksiyonu (Fonksiyon Gösterici parametresi alır)
cv::Mat stereoMatching(const cv::Mat& left, const cv::Mat& right, int windowSize, int maxDisparity, SimilarityFunc metric) {
    int rows = left.rows;
    int cols = left.cols;
    cv::Mat disparityMap = cv::Mat::zeros(left.size(), CV_8UC1);
    int offset = windowSize / 2;

    for (int r = offset; r < rows - offset; ++r) {
        for (int c = offset; c < cols - offset; ++c) {
            double minCost = std::numeric_limits<double>::max();
            int bestDisparity = 0;

            cv::Rect leftRoi(c - offset, r - offset, windowSize, windowSize);
            cv::Mat leftPatch = left(leftRoi);

            for (int d = 0; d < maxDisparity; ++d) {
                if (c - offset - d >= 0) {
                    cv::Rect rightRoi(c - offset - d, r - offset, windowSize, windowSize);
                    cv::Mat rightPatch = right(rightRoi);

                    double cost = metric(leftPatch, rightPatch);
                    if (cost < minCost) {
                        minCost = cost;
                        bestDisparity = d;
                    }
                }
            }
            // Görselleştirmek için derinlik değerini normalize ediyoruz
            disparityMap.at<uchar>(r, c) = static_cast<uchar>((bestDisparity * 255) / maxDisparity);
        }
    }
    return disparityMap;
}

int main() {
    // Görüntüleri gri tonlamalı okuyoruz
    cv::Mat imgLeft = cv::imread("hafta10resim.png", cv::IMREAD_GRAYSCALE);
    cv::Mat imgRight = cv::imread("hafta10resim2.png", cv::IMREAD_GRAYSCALE);

    // Eğer belirtilen isimde dosya yoksa, test için gri görsel oluşturulur
    if (imgLeft.empty() || imgRight.empty()) {
        std::cout << "HATA: hafta10resim.png veya hafta10resim2.png bulunamadı!" << std::endl;
        std::cout << "Test amaçlı yapay görüntüler oluşturuluyor..." << std::endl;
        imgLeft = cv::Mat::ones(150, 200, CV_8UC1) * 128;
        imgRight = cv::Mat::ones(150, 200, CV_8UC1) * 128;
        // Basit bir nesne ekleyelim
        cv::rectangle(imgLeft, cv::Rect(50, 40, 50, 50), cv::Scalar(255), -1);
        cv::rectangle(imgRight, cv::Rect(40, 40, 50, 50), cv::Scalar(255), -1); // 10 piksel disparity
    }

    int windowSize = 9;
    int maxDisparity = 16;

    std::cout << "Stereo eşleştirme başlatılıyor..." << std::endl;

    // SAD Test
    auto start = std::chrono::high_resolution_clock::now();
    cv::Mat dispSAD = stereoMatching(imgLeft, imgRight, windowSize, maxDisparity, computeSAD);
    auto end = std::chrono::high_resolution_clock::now();
    std::cout << "SAD Süresi: " << std::chrono::duration_cast<std::chrono::milliseconds>(end - start).count() << " ms" << std::endl;
    cv::imwrite("disparity_sad.png", dispSAD);

    // SSD Test
    start = std::chrono::high_resolution_clock::now();
    cv::Mat dispSSD = stereoMatching(imgLeft, imgRight, windowSize, maxDisparity, computeSSD);
    end = std::chrono::high_resolution_clock::now();
    std::cout << "SSD Süresi: " << std::chrono::duration_cast<std::chrono::milliseconds>(end - start).count() << " ms" << std::endl;
    cv::imwrite("disparity_ssd.png", dispSSD);

    // ZNCC Test
    start = std::chrono::high_resolution_clock::now();
    cv::Mat dispZNCC = stereoMatching(imgLeft, imgRight, windowSize, maxDisparity, computeZNCC);
    end = std::chrono::high_resolution_clock::now();
    std::cout << "ZNCC Süresi: " << std::chrono::duration_cast<std::chrono::milliseconds>(end - start).count() << " ms" << std::endl;
    cv::imwrite("disparity_zncc.png", dispZNCC);

    std::cout << "İşlem tamamlandı! Sonuç görselleri kaydedildi (disparity_sad.png, disparity_ssd.png, disparity_zncc.png)." << std::endl;
    return 0;
}

Overwriting main.cpp


In [8]:
!g++ main.cpp -o app `pkg-config --cflags --libs opencv4` -I/usr/local/cuda/include -L/usr/local/cuda/lib64 -lcudart
!./app

[ WARN:0@0.089] global ./modules/imgcodecs/src/loadsave.cpp (239) findDecoder imread_('hafta10resim.png'): can't open/read file: check file path/integrity
[ WARN:0@0.089] global ./modules/imgcodecs/src/loadsave.cpp (239) findDecoder imread_('hafta10resim2.png'): can't open/read file: check file path/integrity
HATA: hafta10resim.png veya hafta10resim2.png bulunamadı!
Test amaçlı yapay görüntüler oluşturuluyor...
Stereo eşleştirme başlatılıyor...
SAD Süresi: 145 ms
SSD Süresi: 155 ms
ZNCC Süresi: 357 ms
İşlem tamamlandı! Sonuç görselleri kaydedildi (disparity_sad.png, disparity_ssd.png, disparity_zncc.png).
